# 02 · Exploratory data analysis

**Goal:** understand how the cycle-level data is distributed across the six battery groups, how SOH degrades over time, where temperature values are missing, and whether any records are duplicated, physically implausible or out of temporal order.

All analyses keep `Battery_ID` separate and respect time order. Nothing is imputed or dropped here — problems are only **reported** so later steps can decide how to handle them.

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Resolve the repository root whether the kernel starts in the repo root or in notebooks/.
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "data_pipeline.py").is_file())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src import data_pipeline as dp

DATA_ROOT = dp.resolve_data_root(os.environ.get("DATA_ROOT", "data"), REPO_ROOT)
RESULTS_DIR = REPO_ROOT / "results"

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
print("DATA_ROOT env :", os.environ.get("DATA_ROOT", "<unset, default 'data'>"))
print("DATA_ROOT dir exists:", DATA_ROOT.is_dir())

import matplotlib.pyplot as plt

# Fixed colour per battery group (never re-assigned by rank or filter).
GROUP_COLORS = dict(zip(dp.BATTERY_GROUPS, ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]))
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.color": "#e4e3dd",
    "grid.linewidth": 0.6,
    "axes.edgecolor": "#8a8981",
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "lines.linewidth": 2,
})
FIG_DIR = RESULTS_DIR / "figures"


def savefig(fig, name):
    """Save a figure for the paper/report (results/figures is regenerated, not committed)."""
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

## 2. Load the cycle-level and RPT-level datasets

Only the files needed for EDA are required. The point-level file is not loaded here (see notebook 01 for its chunked profile).

In [ ]:
REQUIRED = [
    "ml_training_dataset.csv",
    "final_ml_dataset.csv",
    "final_battery_health_dataset.csv",
    "final_battery_cycling_features.csv",
]
dp.check_data_files(DATA_ROOT, required=REQUIRED)

ml_train = dp.read_dataset(DATA_ROOT / "ml_training_dataset.csv")
final_ml = dp.read_dataset(DATA_ROOT / "final_ml_dataset.csv")
health = dp.read_dataset(DATA_ROOT / "final_battery_health_dataset.csv")
cycling = dp.read_dataset(DATA_ROOT / "final_battery_cycling_features.csv")
for name, df in [("ml_training", ml_train), ("final_ml", final_ml), ("health", health), ("cycling_features", cycling)]:
    print(f"{name:17s} shape={df.shape}")

# The delivered cycling-features file is point-level (one row per measurement, Date_Time and
# Cycle_Count), not cycle-level as the handover notes say. Its time column is detected here.
CYCLING_TIME = "Cycle_Date" if "Cycle_Date" in cycling else "Date_Time"
CYCLING_INDEX = "Cycle_Index" if "Cycle_Index" in cycling else "Cycle_Count"
print("cycling_features time column:", CYCLING_TIME, "| index column:", CYCLING_INDEX,
      "| median Data_Points per row:", cycling["Data_Points"].median())

## 3. Row counts by battery group

Cycle counts per group across files, and how many `final_ml_dataset.csv` rows carry a known SOH versus those that precede the first RPT (SOH intentionally missing).

In [ ]:
groups = list(dp.BATTERY_GROUPS)
counts = pd.DataFrame({
    "cycling_features_rows": cycling[dp.GROUP_COL].astype(str).value_counts(),
    "final_ml": final_ml[dp.GROUP_COL].astype(str).value_counts(),
    "final_ml_soh_known": final_ml.dropna(subset=["SOH_Percent"])[dp.GROUP_COL].astype(str).value_counts(),
    "ml_training": ml_train[dp.GROUP_COL].astype(str).value_counts(),
    "rpt_measurements": health[dp.GROUP_COL].astype(str).value_counts(),
}).reindex(groups).fillna(0).astype(int)
counts["final_ml_soh_missing"] = counts["final_ml"] - counts["final_ml_soh_known"]
counts["share_of_training_rows_%"] = (counts["ml_training"] / counts["ml_training"].sum() * 100).round(1)
display(pd.concat([counts, counts.sum().to_frame("TOTAL").T.round(1)]))

In [ ]:
c = counts
fig, ax = plt.subplots(figsize=(7, 3.6))
ax.barh(groups, c["final_ml_soh_known"], color="#2a78d6", label="SOH known (in ml_training)", height=0.6)
ax.barh(groups, c["final_ml_soh_missing"], left=c["final_ml_soh_known"] + c["final_ml"].max() * 0.004,
        color="#b5b4ac", label="before first RPT (SOH missing)", height=0.6)
for g, total in zip(groups, c["final_ml"]):
    ax.text(total * 1.01, g, f"{int(total):,}", va="center", fontsize=9, color="#3d3d3a")
ax.invert_yaxis()
ax.set_xlabel("cycles in final_ml_dataset.csv")
ax.set_title("Cycle rows per battery group")
ax.grid(axis="y", visible=False)
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.22), ncol=2, fontsize=8)
plt.tight_layout()
savefig(fig, "02_rows_per_group")
plt.show()

## 4. SOH distributions

SOH is measured only at RPTs; cycle-level SOH in `ml_training_dataset.csv` repeats the most recent RPT value, so its distribution is the RPT distribution weighted by how many cycles fall in each RPT interval. Both views are shown.

In [ ]:
soh_rpt = health.groupby(health[dp.GROUP_COL].astype(str))["SOH_Percent"].describe().reindex(groups)
soh_cycle = ml_train.groupby(ml_train[dp.GROUP_COL].astype(str))["SOH_Percent"].describe().reindex(groups)
display(pd.concat({"RPT-level (health)": soh_rpt, "cycle-level (ml_training)": soh_cycle}, axis=1).round(2))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, (label, df) in zip(axes, [("RPT-level SOH", health), ("Cycle-level SOH (ml_training)", ml_train)]):
    data = [df.loc[df[dp.GROUP_COL].astype(str) == g, "SOH_Percent"].dropna() for g in groups]
    bp = ax.boxplot(data, patch_artist=True, widths=0.5, medianprops={"color": "#1a1a19"})
    for patch, g in zip(bp["boxes"], groups):
        patch.set_facecolor(GROUP_COLORS[g]); patch.set_alpha(0.8); patch.set_edgecolor("white")
    ax.set_xticks(range(1, len(groups) + 1), groups)
    ax.set_title(label)
    ax.grid(axis="x", visible=False)
axes[0].set_ylabel("SOH (%)")
plt.tight_layout()
savefig(fig, "02_soh_distributions")
plt.show()

## 5. Degradation over time

SOH at each RPT per battery, against calendar date (small multiples — one panel per group) and against days since the battery's first RPT (shared axis for comparison). A least-squares slope gives a simple fade rate in SOH percentage points per 100 days.

In [ ]:
h = health.assign(**{dp.GROUP_COL: health[dp.GROUP_COL].astype(str)}).sort_values([dp.GROUP_COL, "Test_Date"])
h["days_since_first_rpt"] = (h["Test_Date"] - h.groupby(dp.GROUP_COL)["Test_Date"].transform("min")).dt.days

fig, axes = plt.subplots(2, 3, figsize=(11, 5.5), sharey=True)
for ax, g in zip(axes.ravel(), groups):
    sub = h[h[dp.GROUP_COL] == g]
    ax.plot(sub["Test_Date"], sub["SOH_Percent"], marker="o", markersize=5, color=GROUP_COLORS[g])
    ax.set_title(g)
    ax.tick_params(axis="x", labelrotation=30, labelsize=8)
for ax in axes[:, 0]:
    ax.set_ylabel("SOH (%)")
fig.suptitle("SOH at each RPT, by battery group")
plt.tight_layout()
savefig(fig, "02_soh_per_group")
plt.show()

fig, ax = plt.subplots(figsize=(8, 3.8))
for g in groups:
    sub = h[h[dp.GROUP_COL] == g]
    if sub.empty:
        continue
    ax.plot(sub["days_since_first_rpt"], sub["SOH_Percent"], marker="o", markersize=4, color=GROUP_COLORS[g], label=g)
ax.set_xlabel("days since first RPT")
ax.set_ylabel("SOH (%)")
ax.set_title("SOH trajectories on a common time base")
ax.legend(frameon=False, ncol=6, loc="lower left", fontsize=8)
plt.tight_layout()
savefig(fig, "02_soh_trajectories")
plt.show()

def fade_rate(sub):
    sub = sub.dropna(subset=["SOH_Percent", "days_since_first_rpt"])
    if len(sub) < 2 or sub["days_since_first_rpt"].nunique() < 2:
        return np.nan
    return np.polyfit(sub["days_since_first_rpt"], sub["SOH_Percent"], 1)[0] * 100

fade = h.groupby(dp.GROUP_COL).apply(
    lambda s: pd.Series({
        "n_rpt": len(s),
        "span_days": s["days_since_first_rpt"].max(),
        "last_SOH_%": s.sort_values("Test_Date")["SOH_Percent"].iloc[-1],
        "min_SOH_%": s["SOH_Percent"].min(),
        "fade_pp_per_100_days": fade_rate(s),
        "non_monotonic_steps": int((s.sort_values("Test_Date")["SOH_Percent"].diff() > 0).sum()),
    }),
    include_groups=False,
).reindex(groups)
fade.round(3)

`non_monotonic_steps` counts RPTs where SOH *increased* versus the previous RPT (measurement noise, capacity recovery after rest, or a change in test conditions). These are kept as measured.

## 6. Missing temperature patterns

Temperatures are **not** imputed in the handover data. Here we measure: how much is missing per group, whether the three temperature columns go missing together, whether missingness comes in contiguous runs (systematic, e.g. a sensor outage) or isolated cycles, and whether it differs between rows with and without a known SOH.

In [ ]:
temp_cols = [c for c in dp.TEMPERATURE_COLUMNS if c in final_ml]
fm = final_ml.assign(**{dp.GROUP_COL: final_ml[dp.GROUP_COL].astype(str)}).sort_values(
    [dp.GROUP_COL, "Cycle_Date", "Cycle_Index"], kind="mergesort"
)
temp_missing = dp.missing_by_group(fm, temp_cols).reindex(groups)
display(temp_missing)

n_missing = fm[temp_cols].isna().sum(axis=1)
pattern = pd.crosstab(fm[dp.GROUP_COL], n_missing.map({0: "none", len(temp_cols): "all"}).fillna("partial").rename("temperature_columns_missing")).reindex(groups).fillna(0).astype(int)
print("How many of the temperature columns are missing per row:")
display(pattern)

In [ ]:
def missing_runs(flags: pd.Series) -> pd.Series:
    """Number of contiguous runs of True and the longest run length."""
    f = flags.to_numpy()
    if not f.any():
        return pd.Series({"runs": 0, "longest_run": 0, "mean_run": 0.0})
    starts = np.flatnonzero(f & ~np.r_[False, f[:-1]])
    ends = np.flatnonzero(f & ~np.r_[f[1:], False])
    lengths = ends - starts + 1
    return pd.Series({"runs": len(lengths), "longest_run": lengths.max(), "mean_run": lengths.mean()})

any_temp_missing = fm[temp_cols].isna().any(axis=1)
runs = any_temp_missing.groupby(fm[dp.GROUP_COL]).apply(missing_runs).unstack().reindex(groups)
runs["missing_rows"] = any_temp_missing.groupby(fm[dp.GROUP_COL]).sum().reindex(groups)

by_label = (
    any_temp_missing.groupby([fm[dp.GROUP_COL], fm["SOH_Percent"].notna().map({True: "SOH known", False: "pre-RPT"})])
    .mean().mul(100).round(2).unstack().reindex(groups)
)
display(pd.concat([runs, by_label.add_prefix("missing_%_")], axis=1))

In [ ]:
fm["month"] = fm["Cycle_Date"].dt.to_period("M")
monthly = any_temp_missing.groupby([fm[dp.GROUP_COL], fm["month"]]).mean().mul(100).unstack(0).reindex(columns=groups)

fig, ax = plt.subplots(figsize=(9, 3.6))
for g in groups:
    if g in monthly and monthly[g].notna().any():
        s = monthly[g].dropna()
        ax.plot(s.index.to_timestamp(), s.values, marker="o", markersize=4, color=GROUP_COLORS[g], label=g)
ax.set_ylabel("% cycles with missing temperature")
ax.set_title("Missing temperature by month and battery group")
ax.legend(frameon=False, ncol=6, fontsize=8)
plt.tight_layout()
savefig(fig, "02_missing_temperature_monthly")
plt.show()
monthly.round(1)

## 7. What a cycle-level row is, and do its features track SOH?

The handover's cycle-level rows come from monthly cycling blocks between RPTs, each split into a few rows. The table shows how many rows share one RPT interval and how many raw measurements a row aggregates. The heatmap shows, **within each battery**, the Spearman rank correlation of every model input with SOH, next to elapsed time. SOH falls almost monotonically with time, so a feature that tracks health should correlate about as strongly as time does. A feature that correlates weakly carries little health information, whatever a model does with it.

In [ ]:
rows_per_interval = ml_train.groupby([dp.GROUP_COL, "RPT_Number"]).size().groupby(dp.GROUP_COL).agg(["mean", "min", "max"])
points = ml_train.groupby(dp.GROUP_COL)["Data_Points"].describe()[["min", "50%", "max"]]
display(pd.concat({"rows per RPT interval": rows_per_interval, "raw points per row": points}, axis=1).reindex(groups).round(1))

feats = dp.model_feature_columns(ml_train)
tr = ml_train.assign(days=(ml_train["Cycle_Date"] - ml_train.groupby(dp.GROUP_COL)["Cycle_Date"].transform("min")).dt.days)
corr = pd.DataFrame({
    g: sub[["days", *feats, "SOH_Percent"]].corr(method="spearman")["SOH_Percent"].drop("SOH_Percent")
    for g, sub in tr.groupby(tr[dp.GROUP_COL].astype(str))
}).reindex(columns=groups)
corr = corr.loc[["days", *corr.drop(index="days").abs().median(axis=1).sort_values(ascending=False).index]]
display(corr.round(2))

fig, ax = plt.subplots(figsize=(6.5, 6.5))
im = ax.imshow(corr.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(groups)), groups)
ax.set_yticks(range(len(corr)), ["elapsed days (reference)" if i == "days" else i for i in corr.index], fontsize=8)
for (i, j), v in np.ndenumerate(corr.to_numpy()):
    ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7, color="white" if abs(v) > 0.6 else "#1a1a19")
ax.axhline(0.5, color="#1a1a19", linewidth=1)
ax.set_title("Within-battery Spearman correlation with SOH", loc="left")
ax.grid(False)
fig.colorbar(im, ax=ax, shrink=0.7, label="Spearman ρ")
plt.tight_layout()
savefig(fig, "02_feature_soh_correlation")
plt.show()
print("median |ρ| of model inputs:", round(float(corr.drop(index="days").abs().median(axis=1).median()), 2),
      "| elapsed days:", round(float(corr.loc["days"].median()), 2))

## 8. Duplicate, invalid and out-of-order records

* **Duplicates:** exact duplicate rows and repeated keys `(Battery_ID, Cycling_Test, Cycle_Index)` for cycle-level files, `(Battery_ID, RPT)` for RPTs.
* **Out of order:** `Cycle_Date` going backwards within a battery in file order, or within a `(Battery_ID, Cycling_Test)` segment when sorted by `Cycle_Index`; for RPTs, `Test_Date` must increase with `RPT_Number`.
* **Invalid:** values outside loose physical-plausibility bounds (`dp.PLAUSIBILITY_BOUNDS`), e.g. voltage outside 0–5 V, efficiency > 1.5, SOH > 150 %.

In [ ]:
CYCLE_KEY = [dp.GROUP_COL, "Cycling_Test", "Cycle_Index"]
dup = pd.DataFrame({
    "cycling_features": dp.find_duplicates(cycling, [dp.GROUP_COL, "Cycling_Test", CYCLING_TIME]),
    "final_ml": dp.find_duplicates(final_ml, CYCLE_KEY),
    "ml_training": dp.find_duplicates(ml_train, CYCLE_KEY),
    "health": dp.find_duplicates(health, [dp.GROUP_COL, "RPT"]),
}).T
display(dup)

order = pd.concat({
    "cycling_features": dp.find_out_of_order(cycling, time_col=CYCLING_TIME, index_col=CYCLING_INDEX),
    "final_ml": dp.find_out_of_order(final_ml),
    "ml_training": dp.find_out_of_order(ml_train),
}, axis=1)
display(order)

hs = health.sort_values([dp.GROUP_COL, "RPT_Number"], kind="mergesort")
rpt_order = hs.groupby(hs[dp.GROUP_COL].astype(str)).apply(
    lambda s: pd.Series({
        "rpt_count": len(s),
        "test_date_decreases_with_rpt_number": int((s["Test_Date"].diff().dt.total_seconds() < 0).sum()),
        "same_day_rpts": int(s["Test_Date"].duplicated().sum()),
        "missing_test_date": int(s["Test_Date"].isna().sum()),
    }),
    include_groups=False,
)
rpt_order

In [ ]:
invalid = pd.concat({
    "final_ml": dp.find_invalid_values(final_ml),
    "ml_training": dp.find_invalid_values(ml_train),
    "health": dp.find_invalid_values(health),
})
invalid

## 9. Summary

In [ ]:
eda_summary = pd.DataFrame(
    [
        ("Battery groups in final_ml", sorted(fm[dp.GROUP_COL].unique())),
        ("Cycles in final_ml / ml_training", f"{len(final_ml):,} / {len(ml_train):,}"),
        ("final_ml rows before first RPT (SOH missing)", int(final_ml["SOH_Percent"].isna().sum())),
        ("Rows with any temperature missing (final_ml)", f"{int(any_temp_missing.sum()):,} ({any_temp_missing.mean() * 100:.2f} %)"),
        ("Groups with temperature gaps", [g for g in groups if runs.loc[g, "missing_rows"] > 0]),
        ("Exact duplicate rows (cycle-level files)", int(dup.loc[["cycling_features", "final_ml", "ml_training"], "exact_duplicate_rows"].sum())),
        ("Duplicate cycle keys (cycle-level files)", int(dup.loc[["cycling_features", "final_ml", "ml_training"], "duplicate_keys"].sum())),
        ("Out-of-order cycle dates (by index, all files)", int(order.xs("date_decreases_by_index", axis=1, level=1).sum().sum())),
        ("RPT dates out of order vs RPT_Number", int(rpt_order["test_date_decreases_with_rpt_number"].sum())),
        ("Values outside plausibility bounds", int(invalid["flagged"].sum())),
        ("RPT-to-RPT SOH increases", int(fade["non_monotonic_steps"].sum())),
        ("Median within-battery |ρ| of model inputs with SOH", round(float(corr.drop(index="days").abs().median(axis=1).median()), 2)),
        ("Within-battery ρ of elapsed days with SOH (median)", round(float(corr.loc["days"].median()), 2)),
    ],
    columns=["finding", "value"],
)
eda_summary

### Interpretation notes and open questions

* **Label granularity.** Cycle-level SOH is a step function that changes only at RPTs. A model therefore sees many cycles sharing one label; per-group metrics and time-aware splits (notebook 04) are required to avoid optimistic scores.
* **Temperature gaps.** If `longest_run` is large relative to `runs`, missingness is systematic (sensor/logging outage) rather than random, and simple mean imputation would bias those periods. The handling choice (native NaN support, median + missing indicator, masking in the TFT) is made in notebooks 03/04 and must be fitted on training rows only.
* **Duplicates / ordering / plausibility flags** are reported, not removed. Any non-zero count above should be confirmed with the data owner before rows are dropped.
* **File order vs. time order.** `date_decreases_in_file_order` > 0 only means the CSV is not stored chronologically; all downstream code sorts by `Battery_ID, Cycle_Date, Cycle_Index` explicitly.
* **Feature informativeness (section 7).** If the model inputs correlate far more weakly with SOH than elapsed time does, the cycle-level features carry little health information, and any model trained on them mostly learns a time trend. Health indicators must then be derived from the point-level data (notebook 10).